# 01 - Coleta: Segurança Pública em Recife

Baixa os microdados oficiais da SDS-PE e coleta, via API do Fogo Cruzado, ocorrências georreferenciadas de tiroteios/disparos em Recife no ano-base de 2025.

In [ ]:
import os
import time
from pathlib import Path
from typing import Any

import pandas as pd
import requests

In [ ]:
def find_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / '.git').exists():
            return p
    return Path.cwd()


ROOT = find_root()
RAW_DIR = ROOT / 'data' / 'raw' / 'seguranca'
RAW_DIR.mkdir(parents=True, exist_ok=True)

YEAR = 2025
CSV_SEPARATOR = ','
OUTPUT_FOGO = RAW_DIR / f'fogo_cruzado_recife_{YEAR}.csv'
CVLI_PATH = RAW_DIR / 'microdados_cvli.xlsx'
CVP_PATH = RAW_DIR / 'microdados_cvp.xlsx'

print(f'Raiz do projeto: {ROOT}')
print(f'Diretório de dados brutos: {RAW_DIR}')

## 1 - Microdados oficiais da SDS-PE

A SDS-PE disponibiliza microdados de CVLI e CVP no nível de município. Eles são usados como contexto oficial da dimensão de segurança.

In [ ]:
SDS_CVLI_URL = (
    'https://www.sds.pe.gov.br/images/indicadores/CVP/'
    'MICRODADOS_DE_CVLI_JAN_2004_A_ABR_2026.xlsx'
)
SDS_CVP_URL = (
    'https://www.sds.pe.gov.br/images/ESTAT%C3%8DSTICAS/GACE/'
    'Microdados_de_CVP_-_Dispon%C3%ADvel_janeiro_de_2014_a_abril_de_2026.xlsx'
)


def download_file(url: str, output_path: Path, overwrite: bool = False):
    if output_path.exists() and not overwrite:
        print(f'Arquivo já existe: {output_path.name}')
        return

    print(f'Baixando: {output_path.name}')
    response = requests.get(url, timeout=120)
    response.raise_for_status()
    output_path.write_bytes(response.content)
    print(f'Arquivo salvo: {output_path}')


download_file(SDS_CVLI_URL, CVLI_PATH)
download_file(SDS_CVP_URL, CVP_PATH)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "openpyxl"], check=True)

In [ ]:
# Validação rápida dos arquivos oficiais baixados
cvli = pd.read_excel(CVLI_PATH, sheet_name='Plan1')
cvp = pd.read_excel(CVP_PATH, sheet_name='microdados cvp')

print(f'CVLI - linhas: {len(cvli)} | colunas: {cvli.columns.tolist()}')
print(f'CVP  - linhas: {len(cvp)} | colunas: {cvp.columns.tolist()}')

cvli_recife_2025 = cvli[(cvli['MUNICIPIO'].str.upper() == 'RECIFE') & (cvli['ANO'] == YEAR)]
cvp_recife_2025 = cvp[(cvp['MUNICÍPIO'].str.upper() == 'RECIFE') & (cvp['ANO'] == YEAR)]

print(f'CVLI Recife {YEAR}: {cvli_recife_2025["TOTAL DE VITIMAS"].sum():,.0f} vítimas')
print(f'CVP Recife {YEAR}: {cvp_recife_2025["TOTAL"].sum():,.0f} ocorrências')
display(cvli_recife_2025.head(3).reset_index(drop=True).style.hide(axis='index'))
display(cvp_recife_2025.head(3).reset_index(drop=True).style.hide(axis='index'))

## 2 - Ocorrências por bairro via Fogo Cruzado

Como a entrega precisa de nota por bairro e mapa de hotspots, a base operacional usada para bairro/geolocalização é a API do Fogo Cruzado.

In [ ]:
API_URL = 'https://api-service.fogocruzado.org.br/api/v2'
PUBLIC_API_SECRET = 'REMOVED'

PERNAMBUCO_ID = '813ca36b-91e3-4a18-b408-60b27a1942ef'
RECIFE_ID = 'fb1c4e7d-1f61-4a86-b514-d93d533df7a3'
API_SECRET = os.getenv('FOGO_CRUZADO_API_SECRET', PUBLIC_API_SECRET)


def get_json(endpoint: str, params: dict[str, Any]) -> dict[str, Any]:
    response = requests.get(
        f'{API_URL}/{endpoint}',
        params=params,
        headers={'x-api-secret': API_SECRET},
        timeout=60,
    )
    response.raise_for_status()
    payload = response.json()
    if payload.get('code', 200) >= 400:
        raise RuntimeError(payload)
    return payload


def nested_name(value, key='name'):
    return str(value.get(key) or '') if isinstance(value, dict) else ''


def summarize_victims(victims):
    dead = wounded = civilians_dead = civilians_wounded = agents_dead = agents_wounded = 0
    for victim in victims:
        situation = str(victim.get('situation') or '').lower()
        person_type = str(victim.get('personType') or '').lower()
        is_dead = situation == 'dead'
        is_wounded = situation == 'wounded'
        is_agent = 'agent' in person_type

        dead += int(is_dead)
        wounded += int(is_wounded)
        civilians_dead += int(is_dead and not is_agent)
        civilians_wounded += int(is_wounded and not is_agent)
        agents_dead += int(is_dead and is_agent)
        agents_wounded += int(is_wounded and is_agent)

    return {
        'mortos': dead,
        'feridos': wounded,
        'baleados': dead + wounded,
        'civis_mortos': civilians_dead,
        'civis_feridos': civilians_wounded,
        'agentes_mortos': agents_dead,
        'agentes_feridos': agents_wounded,
    }


def flatten_occurrence(item):
    context = item.get('contextInfo') or {}
    victims = item.get('victims') or []
    complementary = context.get('complementaryReasons') or []
    clippings = context.get('clippings') or []

    return {
        'id': item.get('id'),
        'documento': item.get('documentNumber'),
        'endereco': item.get('address'),
        'estado': nested_name(item.get('state')),
        'regiao': nested_name(item.get('region'), 'region'),
        'cidade': nested_name(item.get('city')),
        'bairro_original': nested_name(item.get('neighborhood')),
        'sub_bairro': nested_name(item.get('subNeighborhood')),
        'localidade': nested_name(item.get('locality')),
        'latitude': item.get('latitude'),
        'longitude': item.get('longitude'),
        'data_ocorrencia': item.get('date'),
        'acao_policial': bool(item.get('policeAction')),
        'presenca_agente': bool(item.get('agentPresence')),
        'motivo_principal': nested_name(context.get('mainReason')),
        'motivos_complementares': ' | '.join(nested_name(v) for v in complementary),
        'recortes': ' | '.join(nested_name(v) for v in clippings),
        'chacina': bool(context.get('massacre')),
        'unidade_policial': context.get('policeUnit') or '',
        **summarize_victims(victims),
    }

In [ ]:
all_rows = []
page = 1
take = 200

while True:
    payload = get_json(
        'occurrences',
        params={
            'initialdate': f'{YEAR}-01-01',
            'finaldate': f'{YEAR}-12-31',
            'idState': PERNAMBUCO_ID,
            'idCities': RECIFE_ID,
            'typeOccurrence': 'all',
            'page': page,
            'take': take,
        },
    )

    data = payload.get('data') or []
    all_rows.extend(flatten_occurrence(item) for item in data)

    page_meta = payload.get('pageMeta') or {}
    print(f'Página {page}: {len(data)} registros')
    if not page_meta.get('hasNextPage'):
        break

    page += 1
    time.sleep(0.15)

df = pd.DataFrame(all_rows)
df.to_csv(OUTPUT_FOGO, index=False, sep=CSV_SEPARATOR, encoding='utf-8')

print(f'Ocorrências coletadas: {len(df)}')
print(f'Arquivo salvo: {OUTPUT_FOGO}')
display(df.head(10).reset_index(drop=True).style.hide(axis='index'))

In [ ]:
# Conferência inicial de bairros e motivos mais frequentes
print(f'Bairros informados na API: {df["bairro_original"].nunique()}')
display(
    df['bairro_original']
    .value_counts()
    .head(15)
    .rename_axis('bairro_original')
    .reset_index(name='ocorrencias')
    .style.hide(axis='index')
)

display(
    df['motivo_principal']
    .value_counts()
    .rename_axis('motivo_principal')
    .reset_index(name='ocorrencias')
    .style.hide(axis='index')
)